In [ ]:
from datetime import datetime
import pandas as pd
import plotly.graph_objects as go
import pytz

In [ ]:
# Load the dataset
apps_df = pd.read_csv('Play Store Data.csv')
reviews_df = pd.read_csv('User Reviews.csv')
apps_df.head()

In [ ]:
# Drop missing values
apps_df = apps_df.dropna(subset=['App', 'Category', 'Type', 'Price', 'Installs', 'Size', 'Android Ver', 'Content Rating']).copy()

# Clean the 'Installs' column by removing '+' and ',' characters and converting it to integer
apps_df['Installs'] = apps_df['Installs'].str.replace('+', '', regex=False).str.replace(',', '', regex=False)
apps_df = apps_df[apps_df['Installs'].str.isnumeric()].copy()
apps_df['Installs'] = apps_df['Installs'].astype(int)

# Clean the 'Price' column by removing '$' and converting it to float
apps_df['Price'] = apps_df['Price'].astype(str).str.replace('$', '', regex=False)
apps_df['Price'] = apps_df['Price'].astype(float)

# Calculate Revenue
apps_df['Revenue'] = apps_df['Installs'] * apps_df['Price']

apps_df.head()

In [ ]:
def parse_size_to_mb(val):
  val = str(val).strip()

  # Handle Megabytes (e.g., '20M')
  if val.endswith('M') or val.endswith('m'):
    try:
      return float(val[:-1])
    except:
      return None

  # Handle Kilobytes (e.g., '20000k' -> 19.53 MB)
  elif val.endswith('k') or val.endswith('K'):
    try:
      return float(val[:-1]) / 1024.0
    except:
      return None

  # Handles 'Varies with device' and non-numeric values
  return None

def is_android_gt_4(val):
  val = str(val).strip()
  # Pick the first word before space (e.g. '4.0.3' from '4.0.3 and up')
  first_word = val.split(' ')[0]

  # Break the word into pieces by dot
  parts = first_word.split('.')

  # Collect all numeric parts step by step
  version_numbers = []
  for p in parts:
    if p.isdigit():
      version_numbers.append(int(p))

  # Compare with [4, 0]
  if version_numbers > [4, 0]:
    return True
  else:
    return False


# Filter logic
cond_installs = apps_df['Installs'] >= 10000
cond_revenue = (apps_df['Type'] == 'Free') | (apps_df['Revenue'] >= 10000) # Keeping the free apps because it's necessary for free vs paid comparison
cond_content_rating = apps_df['Content Rating'] == 'Everyone'
cond_name_len = apps_df['App'].astype(str).str.len() <= 30
cond_size = apps_df['Size'].apply(parse_size_to_mb) > 15
cond_android = apps_df['Android Ver'].apply(is_android_gt_4)
cond_type = apps_df['Type'].isin(['Free', 'Paid'])

# Apply all filters
filtered_df = apps_df[
    cond_installs
    & cond_revenue
    & cond_content_rating
    & cond_name_len
    & cond_size
    & cond_android
    & cond_type
].copy()

# Inspect filtered dataset
filtered_df[['App', 'Category', 'Content Rating', 'Type', 'Installs', 'Revenue', 'Size', 'Android Ver']]

In [ ]:
# Identify Top 3 categories by total installs from filtered data
top3_categories = (
    filtered_df.groupby('Category')['Installs']
    .sum()
    .nlargest(3)
    .index.tolist()
)

# Filter dataset to include only the Top 3 categories
top3_df = filtered_df[filtered_df['Category'].isin(top3_categories)].copy()

# Calculate average Installs and average Revenue grouped by Category and Type
agg_df = (
    top3_df.groupby(['Category', 'Type'])[['Installs', 'Revenue']]
    .mean()
    .reset_index()
)

# Display the aggregated DataFrame
agg_df.head(10)

In [ ]:
# Check time (1:00 PM to 2:00 PM IST)
ist = pytz.timezone('Asia/Kolkata')
current_time = datetime.now(ist).time()

start_time = datetime.strptime('13:00:00', '%H:%M:%S').time()
end_time = datetime.strptime('14:00:00', '%H:%M:%S').time()

is_within_time = start_time <= current_time < end_time

# Prepare HTML table rows
cat_rows_html = ''
for idx, row in agg_df.iterrows():
  cat_rows_html += f"<tr><td>{row['Category']}</td><td>{row['Type']}</td><td>{int(row['Installs']):,}</td><td>${row['Revenue']:,.2f}</td></tr>\n"

# Defined file paths
graph_path = 'task_3_graph.html'
dashboard_path = 'task_3_dashboard.html'

In [ ]:
# Generating the graph using Plotly and saving it as task_3_graph.html silently if within the allowed time window
if is_within_time:
  fig = go.Figure()

  free_data = agg_df[agg_df['Type'] == 'Free']
  paid_data = agg_df[agg_df['Type'] == 'Paid']

  # Average Installs - Free & Paid
  fig.add_trace(
      go.Bar(
          x=free_data['Category'],
          y=free_data['Installs'],
          name='Free - Avg Installs',
          yaxis='y1',
          offsetgroup='free_installs',
      )
  )

  fig.add_trace(
      go.Bar(
          x=paid_data['Category'],
          y=paid_data['Installs'],
          name='Paid - Avg Installs',
          yaxis='y1',
          offsetgroup='paid_installs',
      )
  )

  # Revenue - Free & Paid
  fig.add_trace(
      go.Scatter(
          x=free_data['Category'],
          y=free_data['Revenue'],
          name='Free - Avg Revenue',
          mode='markers',
          yaxis='y2',
          marker=dict(size=10),
      )
  )

  fig.add_trace(
      go.Scatter(
          x=paid_data['Category'],
          y=paid_data['Revenue'],
          name='Paid - Avg Revenue',
          mode='markers',
          yaxis='y2',
          marker=dict(size=10),
      )
  )

  fig.update_layout(
      title=dict(
          text='Average Installs and Revenue: Free vs Paid Apps',
          x=0.5,
          y=0.95,
          xanchor='center',
          yanchor='top',
      ),
      xaxis=dict(title='Top 3 App Categories'),
      yaxis=dict(
          title='Average Installs',
          type='log',
          side='left',
          tickvals=[100000, 1000000, 10000000, 50000000],
          ticktext=['100K', '1M', '10M', '50M'],
      ),
      yaxis2=dict(title='Average Revenue ($)', overlaying='y', side='right'),
      barmode='group',
      hovermode='x unified',
      legend=dict(
          title='App Type / Metric',
          orientation='h',
          x=0.5,
          y=1.08,
          xanchor='center',
          yanchor='bottom',
      ),
      margin=dict(l=80, r=80, t=120, b=80),
      template='plotly_white',
  )

  # Save graph HTML file silently
  fig.write_html(graph_path)

  status_section_html = f"""
    <div class="card">
        <h2>Graph Status</h2>
        <p style="color: green; font-weight: bold;">Status: Graph Successfully Generated</p>
        <p>The interactive dual-axis chart has been generated and saved to <b>{graph_path}</b> because the execution occurred within the allowed window (1:00 PM - 2:00 PM IST).</p>
        <p><a href="{graph_path}" target="_blank" style="display: inline-block; padding: 10px 15px; background-color: #1a73e8; color: white; text-decoration: none; border-radius: 5px;">Open Graph</a></p>
    </div>
  """

else:
  status_section_html = f"""
    <div class="card">
        <h2>Graph Status</h2>
        <p style="color: red; font-weight: bold;">Status: Graph Restricted</p>
        <p>The interactive graph is only generated and accessible between <b>1:00 PM IST and 2:00 PM IST</b>. The graph file ({graph_path}) was not created. Please rerun the code during the allowed time window.</p>
    </div>
  """


In [ ]:
# Generating the HTML dashboard content
dashboard_html_content = f"""<!DOCTYPE html>
<html>
<head>
    <title>Play Store Installs Dashboard - Task 3</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
            margin: 30px;
            background-color: #f4f6f9;
            color: #333;
        }}
        h1 {{ color: #1a73e8; }}
        .card {{
            background: white;
            padding: 20px;
            margin-bottom: 20px;
            border-radius: 8px;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
        }}
        table {{
            width: 100%;
            border-collapse: collapse;
            margin-top: 10px;
        }}
        th, td {{
            padding: 10px;
            border: 1px solid #ddd;
            text-align: left;
        }}
        th {{ background-color: #f0f4f8; }}
    </style>
</head>
<body>

    <h1>Play Store App Analysis Dashboard (Task 3)</h1>

    <div class="card">
        <h2>Dashboard Summary</h2>
        <p><b>Filter Rules Applied:</b> Installs &ge; 10k, Revenue &ge; $10k (if Paid), Content Rating: Everyone, App Name &le; 30 chars, Size &gt; 15 MB, Android Ver &gt; 4.0.</p>
        <p><b>Total Filtered Apps Analyzed:</b> {len(filtered_df):,}</p>
    </div>

    <div class="card">
        <h2>Top 3 Categories (Free vs Paid Aggregates)</h2>
        <table>
            <tr>
                <th>Category Name</th>
                <th>Type</th>
                <th>Average Installs</th>
                <th>Average Revenue</th>
            </tr>
            {cat_rows_html}
        </table>
    </div>

    {status_section_html}

</body>
</html>
"""

with open(dashboard_path, 'w', encoding='utf-8') as f:
  f.write(dashboard_html_content)